<a href="https://colab.research.google.com/github/snigdha1312/ai-ml-intro-project-soc/blob/main/NLP_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install tensorflow

In [ ]:
import numpy as np
import tensorflow as tf
import random
import pickle
import json

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from keras.models import load_model
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
lemmatizer = WordNetLemmatizer()
intents = json.loads(open("intents (1).json").read())

In [ ]:
words = []
classes = []
documents = []
ignoreLetters = ['?', '!', '.', ',']

In [ ]:
for intent in intents['intents']:
    for pattern in intent['patterns']:
        wordList = nltk.word_tokenize(pattern)
        words.extend(wordList)
        documents.append((wordList, intent['tag']))
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

In [ ]:
words = [lemmatizer.lemmatize(word) for word in words if word not in ignoreLetters]
words = sorted(set(words))
classes = sorted(set(classes))

In [ ]:
pickle.dump(words, open('words.pkl', 'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))

In [ ]:
training = []
outputEmpty = [0]*len(classes)

In [ ]:
for document in documents:
    bag = []
    wordPatterns = document[0]
    wordPatterns = [lemmatizer.lemmatize(word.lower()) for word in wordPatterns]
    for word in words:
        bag.append(1) if word in wordPatterns else bag.append(0)

    outputRow = list(outputEmpty)
    outputRow[classes.index(document[1])] = 1
    training.append(bag + outputRow)

In [ ]:
random.shuffle(training)
training = np.array(training)

In [ ]:
trainX = training[:, :len(words)]
trainY = training[:, len(words):]

In [ ]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(128, input_shape=(len(trainX[0]),), activation = 'relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(64, activation = 'relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(len(trainY[0]), activation = 'softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
sgd = tf.keras.optimizers.SGD(learning_rate = 0.01, momentum = 0.9, nesterov = True)
model.compile(loss = 'categorical_crossentropy', optimizer = sgd, metrics=['accuracy'])

In [ ]:
hist = model.fit(np.array(trainX), np.array(trainY), epochs=200, batch_size=5, verbose =1)
model.save('chatbot_model.h5', hist)
print('Done')

Epoch 1/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.5046e-04 - loss: 5.6123
Epoch 2/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 7.2721e-04 - loss: 5.6079
Epoch 3/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0110 - loss: 5.6004
Epoch 4/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0141 - loss: 5.5784
Epoch 5/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0042 - loss: 5.5620
Epoch 6/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0123 - loss: 5.5526
Epoch 7/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0088 - loss: 5.5337
Epoch 8/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0219 - loss: 5.4763
Epoch 9/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0269 - loss: 5.4187
Epoch 10/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0232 - loss: 5.3707
Epoch 11/200
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0188 - loss: 5.2709
Epoch 12/200
167/167 ━━━━━━━━━━━━

Done


In [ ]:
def clean_up_sentence(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word) for word in sentence_words]
    return sentence_words

In [ ]:
def bag_of_words(sentence):
    sentence_words = clean_up_sentence(sentence)
    bag = [0]*len(words)
    for w in sentence_words:
        for i, word in enumerate(words):
            if word == w:
                bag[i] = 1
    return np.array(bag)

In [ ]:
def predict_class(sentence):
    bow = bag_of_words(sentence)
    res = model.predict(np.array([bow]))[0]
    ERROR_THRESHOLD = 0.25
    results = [[i, r] for i,r in enumerate(res) if r > ERROR_THRESHOLD]

    results.sort(key = lambda x: x[1], reverse = True)
    return_list = []
    for r in results:
        return_list.append({'intent': classes [r[0]], 'probability': str(r[1])})
    return return_list

In [ ]:
def get_response(intents_list, intents_json):
    tag = intents_list[0]['intent']
    list_of_intents = intents_json['intents']
    for i in list_of_intents:
        if i['tag'] == tag:
            result = random.choice (i['responses'])
            break
    return result

print("GO! Bot is running")

GO! Bot is running


In [ ]:
while True:
    message = input("")
    ints = predict_class(message)
    res = get_response(ints, intents)
    print(res)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Hey
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Artificial Superintelligence (ASI) refers to highly autonomous AI systems that surpass human intelligence in all aspects.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
I am a chatbot
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
To stay healthy, maintain a balanced diet, engage in regular physical activity, get enough sleep, manage stress, and avoid harmful habits.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Fitness is essential for overall well-being and can improve your mood, energy levels, and longevity.
